# core

> Core module for lazyq — a lightweight, chainable query pipeline for Python.

In [1]:
#| default_exp core

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export
import csv
import sqlite3

In [4]:
#| export
SKIP = object()  # Sentinel: skip this item in the pipeline
STOP = object() # Sentinel: stop the pipeline immediately

In [5]:
#| export
class Op:
    """Wraps a factory function and a name to lazily create pipeline operations."""
    def __init__(self, factory, name):
        self.name = name
        self.factory = factory
    def __call__(self): return self.factory()

In [6]:
#| export
class Map:
    """Applies a function to each item in the pipeline."""
    def __init__(self, fn): self.fn = fn
    def __call__(self, item): return self.fn(item)

class Filter:
    """Keeps items where fn(item) is True, skips the rest."""
    def __init__(self, fn): self.fn = fn
    def __call__(self, item):
        if not self.fn(item): return SKIP
        return item

class Limit:
    """Stops the pipeline after n items."""
    def __init__(self, n):
        self.n = n
        self.count = 0
    def __call__(self, item):
        if self.count >= self.n: return STOP
        self.count += 1
        return item

class Select:
    """Returns only the specified keys from each dict item."""
    def __init__(self, keys): self.keys = keys if isinstance(keys, list) else [keys]
    def __call__(self, item): return {k: item[k] for k in self.keys}

class GroupBy:
    """Groups items by a key, returning (key, [items]) pairs."""
    def __init__(self, key): self.key = key
    def run(self, data):
        groups = {}
        for item in data:
            k = item[self.key]
            if k not in groups: groups[k] = []
            groups[k].append(item)
        return groups.items()

class Reduce:
    """Reduces all items to a single value using fn."""
    def __init__(self, fn, initial=None):
        self.fn = fn
        self.initial = initial
    def run(self, data):
        it = iter(data)
        if self.initial is None:
            try: acc = next(it)
            except StopIteration: return []
        else: acc = self.initial
        for item in it: acc = self.fn(acc, item)
        return [acc]

class Sort:
    """Sorts all items by a key, optionally in reverse order."""
    def __init__(self, key, reverse=False):
        self.key = key
        self.reverse = reverse
    def run(self, data):
        return sorted(data, key=lambda item: item[self.key] if isinstance(item, dict) else item[self.key], reverse=self.reverse)


In [7]:
#| export
class Condition:
    """A composable condition that can be combined with & for filtering."""
    def __init__(self, fn): self.fn = fn
    def __call__(self, x): return self.fn(x)
    def __and__(self, other): return Condition(lambda x: self(x) and other(x))

class F:
    """A field reference used to build conditions. E.g. F('age') > 18"""
    def __init__(self, name): self.name = name 
    def __eq__(self, value): return Condition(lambda x: x[self.name] == value) # F('age') == 'Alice'
    def __gt__(self, value): return Condition(lambda x: x[self.name] > value) # F('age') > 18
    def __lt__(self, value): return Condition(lambda x: x[self.name] < value) # F('age') < 18
    def not_null(self): return Condition(lambda x: x.get(self.name) is not None) # F('email').not_null()
    def is_null(self): return Condition(lambda x: x.get(self.name) is None) # F('email').is_null()


In [8]:
#| export
def read_csv(path):
    with open(path) as f:
        reader = csv.DictReader(f)
        for row in reader: yield row

def read_sqlite3(db_path, query):
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute(query)
        cols = [col[0] for col in cursor.description]
        for row in cursor: yield dict(zip(cols, row))


In [9]:
#| export
class Query:
    """The main lazy query pipeline. Chain operations and execute only when needed.
    
    Example:
        Query.from_csv('data.csv').filter(F('age') > 18).select('name').collect()
    """
    def __init__(self, data, operations=None):
        self.data = data
        self.operations = operations or []
    def map(self, fn): return Query(self.data, self.operations + [Op(lambda: Map(fn), 'map')])
    def filter(self, fn): return Query(self.data, self.operations + [Op(lambda: Filter(fn), 'filter')])
    def limit(self, n): return Query(self.data, self.operations + [Op(lambda: Limit(n), f'limit({n})')])
    def select(self, keys): return Query(self.data, self.operations + [Op(lambda: Select(keys), f'select({keys})')])
    def groupby(self, key): return GroupedQuery(self.data, self.operations + [Op(lambda: GroupBy(key), f'groupby({key})')])
    def reduce(self, fn, initial=None): return Query(self.data, self.operations + [Op(lambda: Reduce(fn, initial), 'reduce')])
    def sort(self, key, reverse=False):
        name = f'sort({key}, desc)' if reverse else f'sort({key})'
        return Query(self.data, self.operations + [Op(lambda: Sort(key, reverse), name)])
    def sum(self): return Query(self.data, self.operations + [Op(lambda: Reduce(lambda acc, x: acc + x), 'sum')])
    def count(self): return Query(self.data, self.operations + [Op(lambda: Reduce(lambda acc, _: acc + 1, 0), 'count')])
    def min(self): return Query(self.data, self.operations + [Op(lambda: Reduce(lambda acc, x: x if x < acc else acc), 'min')])
    def max(self): return Query(self.data, self.operations + [Op(lambda: Reduce(lambda acc, x: x if x > acc else acc), 'max')])

    def __repr__(self):
        if not self.operations: return f"{self.__class__.__name__}()"
        return f"{self.__class__.__name__}(" + " -> ".join(op.name for op in self.operations) + ")"

    def __iter__(self):
        ops = [op() for op in self.operations]
        data = self.data() if callable(self.data) else self.data
        for op in ops:
            if isinstance(op, (GroupBy, Reduce, Sort)): data = op.run(data)
            else: data = self._apply_stream_op(data, op)
        for item in data: yield item

    def _apply_stream_op(self, data, op):
        for item in data:
            item = op(item)
            if item is STOP: return
            if item is SKIP: continue
            yield item

    def collect(self, n=None): return list(self) if n is None else list(self)[:n]
    def show(self, n=5):
        for i, item in enumerate(self):
            if i == n: break
            print(item)

    @classmethod
    def from_iterable(cls, data): return cls(data)
    @classmethod
    def from_csv(cls, path): return cls(lambda: read_csv(path))
    @classmethod
    def from_sqlite(cls, db_path, table): return cls(lambda: read_sqlite3(db_path, f'SELECT * FROM {table}'))
    @classmethod
    def from_sqlite_query(cls, db_path, query): return cls(lambda: read_sqlite3(db_path, query))

class GroupedQuery(Query):
    """A query pipeline for grouped data, returned by groupby().
    
    Example:
        Query.from_csv('data.csv').groupby('country').count().collect()
    """
    def __init__(self, data, operations=None): super().__init__(data, operations)
    # Count items in each group
    def count(self): return Query(self.data, self.operations + [Op(lambda: Map(lambda g: (g[0], len(g[1]))), 'count')])
    # Sum a field across each group
    def sum(self, field): return Query(self.data, self.operations + [Op(lambda: Map(lambda g: (g[0], sum(int(i[field]) for i in g[1]))), 'sum')])
    # Take first n items from each group
    def take(self, n): return GroupedQuery(self.data, self.operations + [Op(lambda: Map(lambda g: (g[0], g[1][:n])), f"take({n})")])
    # Apply fn to each (key, items) group
    def map(self, fn): return GroupedQuery(self.data, self.operations + [Op(lambda: Map(fn), 'map')])
    def max(self, field=None):
        if field is None: return Query.max()
        else: return self.map(lambda g: (g[0], max(g[1], key=lambda x: x[field])))
    def min(self, field=None):
        if field is None: return Query.min()
        else: return self.map(lambda g: (g[0], min(g[1], key=lambda x: x[field])))


In [10]:
#| hide
import nbdev; nbdev.nbdev_export()